In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import cv2
import os
from tqdm import tqdm
from PIL import Image
import SimpleITK as sitk
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [14]:
# ── Configuration ─────────────────────────────────────────────────────────────
CHECKPOINT_PATH  = './rnnet_916.pth'   # your trained RNNet-MST checkpoint
SUBSET_CSV_PATH  = './dataset_nodule21/cxr_images/proccessed_data/subset_metadata2.csv'
TARGET_SIZE      = 224
BATCH_SIZE       = 16
NUM_WORKERS      = 0

candidate_image_dirs = [
    './dataset_nodule21/cxr_images/proccessed_data/split_data/test/images',
    './dataset_nodule21/cxr_images/proccessed_data/split_data/val/images',
    './dataset_nodule21/cxr_images/proccessed_data/split_data/train/images',
]

def resolve_img_path(img_name):
    for d in candidate_image_dirs:
        p = os.path.join(d, img_name)
        if os.path.exists(p):
            return p
    return None

val_transform = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [15]:
# ── RNNet-MST Model (no spatial attention) ────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=2, mlp_ratio=1.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim), nn.Dropout(dropout)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        if H * W > 196:
            x_down = F.adaptive_avg_pool2d(x, (14, 14))
            H_d, W_d = 14, 14
            x_seq = x_down.flatten(2).transpose(1, 2)
        else:
            x_seq = x.flatten(2).transpose(1, 2)
            H_d, W_d = H, W
        x_norm = self.norm1(x_seq)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x_seq = x_seq + attn_out
        x_seq = x_seq + self.mlp(self.norm2(x_seq))
        x_out = x_seq.transpose(1, 2).reshape(B, C, H_d, W_d)
        if H_d != H or W_d != W:
            x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)
        return x + x_out


class ResNetWithMultiScaleAttention(nn.Module):
    def __init__(self, num_classes=2, pretrained=False, num_heads=2, dropout=0.1, active_stages=[1,2,3,4]):
        super().__init__()
        self.active_stages = active_stages
        backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.act1, backbone.maxpool)
        self.stage1 = backbone.layer1
        self.stage2 = backbone.layer2
        self.stage3 = backbone.layer3
        self.stage4 = backbone.layer4
        self.trans1 = TransformerBlock(256,  num_heads=2,        mlp_ratio=1.0, dropout=dropout) if 1 in active_stages else None
        self.trans2 = TransformerBlock(512,  num_heads=2,        mlp_ratio=1.0, dropout=dropout) if 2 in active_stages else None
        self.trans3 = TransformerBlock(1024, num_heads=num_heads, mlp_ratio=1.0, dropout=dropout) if 3 in active_stages else None
        self.trans4 = TransformerBlock(2048, num_heads=num_heads, mlp_ratio=1.0, dropout=dropout) if 4 in active_stages else None
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        if self.trans1 is not None: x = self.trans1(x)
        x = self.stage2(x)
        if self.trans2 is not None: x = self.trans2(x)
        x = self.stage3(x)
        if self.trans3 is not None: x = self.trans3(x)
        x = self.stage4(x)
        if self.trans4 is not None: x = self.trans4(x)
        x = self.global_pool(x).flatten(1)
        return self.classifier(x)

    def get_attention_map(self, x):
        """
        Extract a spatial attention proxy from the final transformer block
        by taking the mean of the last-stage feature map after trans4.
        Returns a [1, 1, H, W] map normalized to [0, 1].
        """
        with torch.no_grad():
            x = self.stem(x)
            x = self.stage1(x)
            if self.trans1 is not None: x = self.trans1(x)
            x = self.stage2(x)
            if self.trans2 is not None: x = self.trans2(x)
            x = self.stage3(x)
            if self.trans3 is not None: x = self.trans3(x)
            x = self.stage4(x)
            if self.trans4 is not None: x = self.trans4(x)
            # Mean across channels → proxy attention map [B, 1, H, W]
            att = x.mean(dim=1, keepdim=True)
            # Normalize to [0, 1]
            b, c, h, w = att.shape
            att_flat = att.view(b, -1)
            att_min  = att_flat.min(dim=1, keepdim=True)[0].view(b, 1, 1, 1)
            att_max  = att_flat.max(dim=1, keepdim=True)[0].view(b, 1, 1, 1)
            att = (att - att_min) / (att_max - att_min + 1e-8)
        return att

In [16]:
# ── Load checkpoint ───────────────────────────────────────────────────────────
print(f'Loading RNNet-MST from {CHECKPOINT_PATH}...')
model = ResNetWithMultiScaleAttention(num_classes=2, pretrained=False, num_heads=2, dropout=0.1, active_stages=[1,2,3,4])
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()
print('✓ Model loaded and set to eval mode')

Loading RNNet-MST from ./rnnet_916.pth...
✓ Model loaded and set to eval mode


In [17]:
# ── Load subset metadata ──────────────────────────────────────────────────────
subset_df = pd.read_csv(SUBSET_CSV_PATH).copy()
subset_df['resolved_path'] = subset_df['img_name'].apply(resolve_img_path)
missing = subset_df['resolved_path'].isna().sum()
if missing > 0:
    print(f'Warning: dropping {missing} rows with missing image files')
subset_df = subset_df[subset_df['resolved_path'].notna()].reset_index(drop=True)
print(f'Usable rows   : {len(subset_df)}')
print(f'Unique images : {subset_df["img_name"].nunique()}')
print(f'Positive rows : {(subset_df["label"]==1).sum()}')

Usable rows   : 171
Unique images : 135
Positive rows : 171


In [18]:
# ── Compute 4 attention-bbox alignment metrics ─────────────────────────────────
print('\n' + '='*70)
print('Computing Attention-BBox Alignment Metrics on RNNet-MST (no spatial attention)')
print('='*70)

unique_images = subset_df.drop_duplicates(subset='img_name').reset_index(drop=True)

bbox_coverage_list   = []
detection_rate_list  = []
peak_proximity_list  = []
attention_focus_list = []

for idx in tqdm(range(len(unique_images)), desc='Attention Metrics'):
    row   = unique_images.iloc[idx]
    label = int(row['label'])

    if label == 0:
        continue  # only evaluate on positive (nodule) cases

    # ── Load image ────────────────────────────────────────────────────────────
    image_itk = sitk.ReadImage(row['resolved_path'])
    arr = sitk.GetArrayFromImage(image_itk)
    if len(arr.shape) == 3:
        arr = arr[0]
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx > mn:
        arr = ((arr - mn) / (mx - mn) * 255).astype(np.uint8)
    else:
        arr = np.zeros_like(arr, dtype=np.uint8)
    arr = np.stack([arr, arr, arr], axis=-1)
    img_tensor = val_transform(Image.fromarray(arr)).unsqueeze(0).to(device)

    # ── Get attention map ─────────────────────────────────────────────────────
    attention_map = model.get_attention_map(img_tensor)          # [1, 1, 7, 7]
    attention_map = attention_map.squeeze().cpu().numpy()         # [7, 7]
    attention_map = cv2.resize(attention_map, (TARGET_SIZE, TARGET_SIZE))  # [224, 224]

    # Normalize to [0, 1]
    a_min, a_max = attention_map.min(), attention_map.max()
    if a_max > a_min:
        attention_map = (attention_map - a_min) / (a_max - a_min)

    # ── Get all bboxes for this image ─────────────────────────────────────────
    img_name = row['img_name']
    img_rows = subset_df[subset_df['img_name'] == img_name]
    img_w    = img_rows.iloc[0].get('img_width',  1024)
    img_h    = img_rows.iloc[0].get('img_height', 1024)

    img_bbox_coverage   = []
    img_detection_rate  = []
    img_peak_proximity  = []
    img_attention_focus = []

    for _, brow in img_rows.iterrows():
        if brow['label'] != 1:
            continue

        x = (brow['x']     / img_w) * TARGET_SIZE
        y = (brow['y']     / img_h) * TARGET_SIZE
        w = (brow['width'] / img_w) * TARGET_SIZE
        h = (brow['height']/ img_h) * TARGET_SIZE

        x_pix = max(0, int(x))
        y_pix = max(0, int(y))
        x_end = min(TARGET_SIZE, int(x + w))
        y_end = min(TARGET_SIZE, int(y + h))

        bbox_mask = np.zeros((TARGET_SIZE, TARGET_SIZE), dtype=np.float32)
        bbox_mask[y_pix:y_end, x_pix:x_end] = 1.0
        bbox_area = bbox_mask.sum()
        if bbox_area == 0:
            continue

        # 1. BBox Coverage
        coverage = (attention_map * bbox_mask).sum() / bbox_area
        img_bbox_coverage.append(float(coverage))

        # 2. Detection Rate
        high_att      = (attention_map > 0.5).astype(np.float32)
        overlap_ratio = (high_att * bbox_mask).sum() / bbox_area
        img_detection_rate.append(1.0 if overlap_ratio > 0.2 else 0.0)

        # 3. Peak Proximity
        peak_y, peak_x = np.unravel_index(np.argmax(attention_map), attention_map.shape)
        bbox_cx  = x + w / 2
        bbox_cy  = y + h / 2
        dist     = np.sqrt((peak_x - bbox_cx)**2 + (peak_y - bbox_cy)**2)
        max_dist = np.sqrt(TARGET_SIZE**2 + TARGET_SIZE**2)
        proximity = max(0.0, 1.0 - dist / max_dist)
        img_peak_proximity.append(float(proximity))

        # 4. Attention Focus
        outside_mask = 1.0 - bbox_mask
        mean_inside  = (attention_map * bbox_mask).sum()  / (bbox_area + 1e-8)
        mean_outside = (attention_map * outside_mask).sum() / (outside_mask.sum() + 1e-8)
        focus = min(mean_inside / (mean_outside + 1e-8), 10.0)
        img_attention_focus.append(float(focus))

    if img_bbox_coverage:
        bbox_coverage_list.append(np.mean(img_bbox_coverage))
        detection_rate_list.append(np.mean(img_detection_rate))
        peak_proximity_list.append(np.max(img_peak_proximity))
        attention_focus_list.append(np.mean(img_attention_focus))

print('\n' + '='*70)
print('ATTENTION-BBOX ALIGNMENT METRICS — RNNet-MST (no spatial attention)')
print('='*70)
print(f'  Images evaluated : {len(bbox_coverage_list)}')
print(f'  BBox Coverage    : {np.mean(bbox_coverage_list):.4f}')
print(f'  Detection Rate   : {np.mean(detection_rate_list):.4f}  ({np.mean(detection_rate_list)*100:.2f}%)')
print(f'  Peak Proximity   : {np.mean(peak_proximity_list):.4f}')
print(f'  Attention Focus  : {np.mean(attention_focus_list):.4f}')


Computing Attention-BBox Alignment Metrics on RNNet-MST (no spatial attention)


Attention Metrics: 100%|██████████| 135/135 [00:05<00:00, 25.86it/s]


ATTENTION-BBOX ALIGNMENT METRICS — RNNet-MST (no spatial attention)
  Images evaluated : 135
  BBox Coverage    : 0.3414
  Detection Rate   : 0.2852  (28.52%)
  Peak Proximity   : 0.7601
  Attention Focus  : 1.6673
